# Sentinel-5P TROPOMI CH₄ Data Downloader and India Clipping

This notebook downloads Sentinel-5P TROPOMI Level-2 methane files, clips valid pixels to the India boundary, and saves the clipped files to Google Drive.

The workflow is designed for long Colab runs. It uses parallel downloading, sequential NetCDF clipping, batch syncing to Drive, and checkpointing so that the notebook can be stopped and restarted.

**Run order:** run Cells 1–6 for setup and testing, then run Cell 7 for batch processing. Cell 8 is optional quality checking.

## Cell 1 — Install packages

Install the packages required for downloading, reading NetCDF files, clipping pixels, and displaying progress.

In [ ]:
%%capture
!pip install netCDF4 geopandas shapely tqdm requests --quiet

import netCDF4, geopandas, shapely, tqdm, requests
print('✅ dependencies ready | shapely', shapely.__version__)

## Cell 2 — Mount Google Drive

Google Drive is used for the input link file, India boundary shapefile, and final clipped outputs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✅ Drive mounted')

## Cell 3 — Configuration

Set Earthdata login details, input paths, output paths, and processing options.

Before uploading this notebook to GitHub, keep the username and password as placeholders. Do not upload private credentials.

In [ ]:
import os

# ── Earthdata credentials ─────────────────────────────────────
# Replace these only inside your private Colab runtime.
# Do not commit real credentials to GitHub.
EARTHDATA_USER = 'YOUR_EARTHDATA_USERNAME'
EARTHDATA_PASS = 'YOUR_EARTHDATA_PASSWORD'

# ── Paths ─────────────────────────────────────────────────────
LINKS_FILE = '/content/drive/MyDrive/subset_S5P_L2__CH4____HiR_2_20260418_181122_.txt'
INDIA_SHP = '/content/drive/MyDrive/roi/India_International_Boundary.shp'
DRIVE_OUT_DIR = '/content/drive/MyDrive/S5P_CH4_India'

# Local Colab folders. These are cleared regularly during processing.
TMP_DIR = '/content/raw'
OUT_DIR = '/content/clipped'

# ── Processing options ────────────────────────────────────────
N_DL_WORKERS = 4        # parallel downloads
BATCH_SIZE = 25         # sync clipped files to Drive after this many files
QA_THRESHOLD = 0.5      # keep pixels with qa_value >= this threshold
MIN_FREE_GB = 2.5       # pause if Colab disk space goes below this value

for d in (TMP_DIR, OUT_DIR, DRIVE_OUT_DIR):
    os.makedirs(d, exist_ok=True)

print('✅ configuration ready')
print('Output folder:', DRIVE_OUT_DIR)

## Cell 4 — Set Earthdata authentication

This writes a temporary `.netrc` file in the Colab session so that NASA Earthdata links can be downloaded.

In [ ]:
import os

def setup_earthdata_auth(user, pwd):
    netrc_path = os.path.expanduser('~/.netrc')
    cookies_path = os.path.expanduser('~/.urs_cookies')

    if user.startswith('YOUR_') or pwd.startswith('YOUR_'):
        raise ValueError('Please enter your Earthdata username and password in Cell 3 before running this cell.')

    with open(netrc_path, 'w') as f:
        f.write(f'machine urs.earthdata.nasa.gov login {user} password {pwd}
')
    os.chmod(netrc_path, 0o600)

    open(cookies_path, 'w').close()
    print(f'✅ netrc written for {user}')

setup_earthdata_auth(EARTHDATA_USER, EARTHDATA_PASS)

## Cell 5 — Helper functions

This cell defines the link parser, downloader, disk-space checks, and India clipping function. The clipping uses a vectorized point-in-polygon test to avoid slow pixel-by-pixel loops.

In [ ]:
import os, re, gc, time, shutil, threading
import numpy as np
import requests
import netCDF4 as nc
import geopandas as gpd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ── Shapely vectorized point-in-polygon ───────────────────────
try:
    from shapely import contains_xy as _contains_xy  # shapely >= 2.0
    def _points_inside(geom, lons, lats):
        return _contains_xy(geom, lons, lats)
except ImportError:
    from shapely.vectorized import contains as _v_contains
    def _points_inside(geom, lons, lats):
        return _v_contains(geom, lons, lats)

# ── Load India boundary once ──────────────────────────────────
print('Loading India boundary...')
_india_gdf = gpd.read_file(INDIA_SHP).to_crs('EPSG:4326')
_india_geom = _india_gdf.union_all() if hasattr(_india_gdf, 'union_all') else _india_gdf.unary_union
INDIA_MINX, INDIA_MINY, INDIA_MAXX, INDIA_MAXY = _india_geom.bounds
print(f'✅ boundary loaded | bbox: lon[{INDIA_MINX:.2f}, {INDIA_MAXX:.2f}] lat[{INDIA_MINY:.2f}, {INDIA_MAXY:.2f}]')

# ── Thread-local HTTP session with retry ──────────────────────
_local = threading.local()

def _session():
    if not hasattr(_local, 's'):
        s = requests.Session()
        s.auth = (EARTHDATA_USER, EARTHDATA_PASS)
        retry = Retry(
            total=5,
            backoff_factor=2,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=['GET']
        )
        s.mount('https://', HTTPAdapter(max_retries=retry))
        _local.s = s
    return _local.s

# ── Link parser ───────────────────────────────────────────────
def parse_links(path):
    out = []
    with open(path) as f:
        for line in f:
            url = line.strip()
            if not url or 'sentiwiki' in url or url.endswith('.pdf'):
                continue
            m = re.search(r'LABEL=([^&]+)', url)
            if m:
                label = requests.utils.unquote(m.group(1))
                out.append((url, label))
    return out

# ── Disk helpers ──────────────────────────────────────────────
def free_gb(path='/content'):
    return shutil.disk_usage(path).free / (1024**3)

def wait_for_disk(min_gb=MIN_FREE_GB):
    while free_gb() < min_gb:
        print(f'⚠ disk low ({free_gb():.1f} GB), waiting 20 seconds...')
        time.sleep(20)

# ── Download function ─────────────────────────────────────────
def download_file(url, dest, timeout=300):
    with _session().get(url, stream=True, timeout=timeout, allow_redirects=True) as r:
        r.raise_for_status()
        with open(dest, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk:
                    f.write(chunk)
    return dest

# ── Clip Sentinel-5P granule to India ─────────────────────────
def clip_nc4_to_india(raw_path, out_path, qa_thresh=QA_THRESHOLD):
    """Clip an S5P Level-2 CH4 swath file to India. Returns valid pixel count."""
    with nc.Dataset(raw_path, 'r') as src:
        grp = src['PRODUCT']

        lat = np.array(grp['latitude'][0])
        lon = np.array(grp['longitude'][0])
        qa = np.array(grp['qa_value'][0])
        ch4 = np.array(grp['methane_mixing_ratio'][0])

        has_prec = 'methane_mixing_ratio_precision' in grp.variables
        ch4_prec = np.array(grp['methane_mixing_ratio_precision'][0]) if has_prec else None

        fv_ch4 = grp['methane_mixing_ratio']._FillValue
        fv_qa = grp['qa_value']._FillValue
        g_attrs = {k: src.getncattr(k) for k in src.ncattrs()}

        # QA and valid-value filter
        valid = (qa >= qa_thresh) & (ch4 != fv_ch4) & np.isfinite(ch4) & (ch4 > 0)

        # Fast bounding-box pre-filter
        bbox = (
            (lat >= INDIA_MINY) & (lat <= INDIA_MAXY) &
            (lon >= INDIA_MINX) & (lon <= INDIA_MAXX)
        )
        cand = valid & bbox
        if not cand.any():
            return 0

        # Exact point-in-polygon test for candidate pixels
        inside = _points_inside(_india_geom, lon[cand], lat[cand])
        final = np.zeros_like(cand)
        final[cand] = inside

        n = int(final.sum())
        if n == 0:
            return 0

        # Write clipped output as 1-D arrays. Compression is avoided for safer long runs.
        with nc.Dataset(out_path, 'w', format='NETCDF4') as dst:
            dst.setncatts(g_attrs)
            dst.clipping_boundary = 'India_International_Boundary EPSG:4326'
            dst.qa_threshold = str(qa_thresh)
            dst.n_valid_pixels = n

            gout = dst.createGroup('PRODUCT')
            gout.createDimension('pixel', n)

            def _w(name, arr, fv, units='', long_name=''):
                v = gout.createVariable(name, 'f4', ('pixel',), fill_value=fv)
                v[:] = arr[final].astype('f4')
                if units:
                    v.units = units
                if long_name:
                    v.long_name = long_name

            _w('latitude', lat, -999., 'degrees_north', 'latitude')
            _w('longitude', lon, -999., 'degrees_east', 'longitude')
            _w('methane_mixing_ratio', ch4, fv_ch4, 'mol mol-1', 'column-averaged dry-air mole fraction of methane')
            _w('qa_value', qa, fv_qa, '1', 'data quality value')

            if ch4_prec is not None:
                _w('methane_mixing_ratio_precision', ch4_prec, fv_ch4, 'mol mol-1', 'methane precision')

    del lat, lon, qa, ch4, ch4_prec, valid, bbox, cand, inside, final
    gc.collect()
    return n

print('✅ helper functions ready')

## Cell 6 — Test one file

Run this cell before the full batch. It downloads one file, clips it to India, and plots the clipped pixels.

In [ ]:
import matplotlib.pyplot as plt
import netCDF4 as nc
import numpy as np
import os, time

all_links = parse_links(LINKS_FILE)
print(f'Found {len(all_links):,} data URLs')

if len(all_links) == 0:
    raise ValueError('No downloadable Sentinel-5P links were found. Check LINKS_FILE.')

# Select one test file.
test_url, test_label = all_links[min(150, len(all_links) - 1)]
print(f'
🔬 Test file: {test_label}')

raw_path = os.path.join(TMP_DIR, test_label)
out_path = os.path.join(OUT_DIR, test_label)

for p in (raw_path, out_path):
    if os.path.exists(p):
        os.remove(p)

t0 = time.time()
print(' ↓ downloading...')
download_file(test_url, raw_path)
print(f' ✓ downloaded ({os.path.getsize(raw_path) / 1e6:.1f} MB)')

print(' ✂ clipping...')
n_pix = clip_nc4_to_india(raw_path, out_path)
os.remove(raw_path)
print(f' ✓ clipped: {n_pix:,} pixels in {time.time() - t0:.1f} seconds')

if n_pix == 0:
    print('
⚠ No pixels inside India for this granule. Try a different index in all_links.')
else:
    with nc.Dataset(out_path, 'r') as ds:
        grp = ds['PRODUCT']
        lats = grp['latitude'][:]
        lons = grp['longitude'][:]
        ch4 = grp['methane_mixing_ratio'][:] * 1e9  # ppb

    vmin, vmax = np.nanpercentile(ch4, [2, 98])

    fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), gridspec_kw={'width_ratios': [2, 1]})

    ax = axes[0]
    _india_gdf.plot(ax=ax, facecolor='#f5f0eb', edgecolor='#555', linewidth=0.8)
    sc = ax.scatter(lons, lats, c=ch4, s=0.5, cmap='RdYlBu_r', vmin=vmin, vmax=vmax, alpha=0.9)
    cb = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.03)
    cb.set_label('CH₄ (ppb)')
    ax.set_xlim(65, 100)
    ax.set_ylim(5, 40)
    ax.set_title(f'S5P CH₄ over India
{test_label[:55]}', fontsize=10)
    ax.set_xlabel('Longitude (°E)')
    ax.set_ylabel('Latitude (°N)')
    ax.grid(alpha=0.3)

    ax2 = axes[1]
    ax2.hist(ch4[np.isfinite(ch4)], bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
    ax2.axvline(np.nanmedian(ch4), color='red', linestyle='--', label=f'Median: {np.nanmedian(ch4):.1f}')
    ax2.set_xlabel('CH₄ (ppb)')
    ax2.set_ylabel('Pixel count')
    ax2.set_title(f'n={len(ch4):,} | QA ≥ {QA_THRESHOLD}')
    ax2.legend()
    ax2.grid(alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()

    print(f'
✅ test OK | CH₄ {vmin:.0f}–{vmax:.0f} ppb | median {np.nanmedian(ch4):.1f} ppb')

## Cell 7 — Batch download and clipping

This cell processes all pending files. It skips files already saved in the Drive output folder. Downloading is parallel, but clipping is sequential to avoid NetCDF/HDF5 write conflicts.

In [ ]:
import queue, gc
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

all_links = parse_links(LINKS_FILE)

done_drive = set(os.listdir(DRIVE_OUT_DIR))
pending = [(u, l) for u, l in all_links if l not in done_drive and (l + '.empty') not in done_drive]

print(f'📋 total {len(all_links):,} | done/marked {len(done_drive):,} | pending {len(pending):,}')
print(f'💾 free disk: {free_gb():.1f} GB
')

if not pending:
    print('🎉 nothing to do')
else:
    dl_queue = queue.Queue(maxsize=N_DL_WORKERS * 2)
    STOP = object()
    stats = {'ok': 0, 'empty': 0, 'error': 0, 'dl_err': 0}
    log_path = os.path.join(DRIVE_OUT_DIR, 'processing_log.txt')

    def download_worker(url, label):
        raw = os.path.join(TMP_DIR, label)
        try:
            wait_for_disk()
            download_file(url, raw)
            return (label, raw, None)
        except Exception as e:
            if os.path.exists(raw):
                try:
                    os.remove(raw)
                except Exception:
                    pass
            return (label, None, str(e))

    def feeder(executor):
        for url, label in pending:
            fut = executor.submit(download_worker, url, label)
            dl_queue.put(fut)
        dl_queue.put(STOP)

    def sync_batch_to_drive():
        moved = 0
        for f in os.listdir(OUT_DIR):
            src = os.path.join(OUT_DIR, f)
            dst = os.path.join(DRIVE_OUT_DIR, f)
            try:
                shutil.move(src, dst)
                moved += 1
            except Exception as e:
                print(f' ! move error {f}: {e}')

        for f in os.listdir(TMP_DIR):
            try:
                os.remove(os.path.join(TMP_DIR, f))
            except Exception:
                pass

        gc.collect()
        return moved

    pbar = tqdm(total=len(pending), desc='Processing', unit='file', ncols=100, dynamic_ncols=False)
    since_sync = 0
    t_start = time.time()

    with ThreadPoolExecutor(max_workers=N_DL_WORKERS) as dl_exec:
        feeder_thread = threading.Thread(target=feeder, args=(dl_exec,), daemon=True)
        feeder_thread.start()

        while True:
            item = dl_queue.get()
            if item is STOP:
                break

            label, raw_path, dl_err = item.result()

            if dl_err is not None:
                stats['dl_err'] += 1
                with open(log_path, 'a') as lg:
                    lg.write(f'DL_ERR	{label}	{dl_err[:200]}
')
                pbar.update(1)
                pbar.set_postfix(**stats, refresh=False)
                continue

            out_path = os.path.join(OUT_DIR, label)

            try:
                n_pix = clip_nc4_to_india(raw_path, out_path)

                if n_pix > 0:
                    stats['ok'] += 1
                    with open(log_path, 'a') as lg:
                        lg.write(f'OK	{label}	{n_pix}
')
                else:
                    stats['empty'] += 1
                    if os.path.exists(out_path):
                        os.remove(out_path)
                    with open(log_path, 'a') as lg:
                        lg.write(f'EMPTY	{label}	0
')
                    marker = os.path.join(DRIVE_OUT_DIR, label + '.empty')
                    open(marker, 'w').close()

            except Exception as e:
                stats['error'] += 1
                with open(log_path, 'a') as lg:
                    lg.write(f'CLIP_ERR	{label}	{str(e)[:200]}
')
                if os.path.exists(out_path):
                    os.remove(out_path)

            finally:
                if raw_path and os.path.exists(raw_path):
                    try:
                        os.remove(raw_path)
                    except Exception:
                        pass

            since_sync += 1
            pbar.update(1)
            pbar.set_postfix(
                ok=stats['ok'],
                empty=stats['empty'],
                err=stats['error'] + stats['dl_err'],
                disk=f'{free_gb():.1f}G',
                refresh=False
            )

            if since_sync >= BATCH_SIZE:
                moved = sync_batch_to_drive()
                since_sync = 0
                pbar.write(
                    f' 💾 synced batch → Drive ({moved} files) | '
                    f'disk {free_gb():.1f}G | elapsed {(time.time() - t_start) / 60:.1f} min'
                )

    moved = sync_batch_to_drive()
    pbar.close()

    elapsed = time.time() - t_start
    print(f'
{"=" * 50}')
    print(f'🏁 DONE in {elapsed / 60:.1f} minutes')
    print(f' ✅ ok     : {stats["ok"]:,}')
    print(f' ⬜ empty  : {stats["empty"]:,} (no India pixels)')
    print(f' ❌ errors : {stats["error"] + stats["dl_err"]:,} (see {log_path})')

    drive_n = len([f for f in os.listdir(DRIVE_OUT_DIR) if f.endswith('.nc4')])
    print(f' 📁 on Drive: {drive_n:,} clipped NetCDF files')

## Cell 8 — Optional coverage map

This optional check samples the clipped files and plots the spatial coverage of available methane pixels over India.

In [ ]:
import matplotlib.pyplot as plt
import netCDF4 as nc
import numpy as np
from tqdm import tqdm
import os

files = sorted(f for f in os.listdir(DRIVE_OUT_DIR) if f.endswith('.nc4'))
print(f'Aggregating from {len(files):,} files...')

lats, lons, ch4s = [], [], []

for f in tqdm(files):
    try:
        with nc.Dataset(os.path.join(DRIVE_OUT_DIR, f)) as ds:
            g = ds['PRODUCT']
            step = max(1, g.dimensions['pixel'].size // 300)
            lats.extend(g['latitude'][::step])
            lons.extend(g['longitude'][::step])
            ch4s.extend(g['methane_mixing_ratio'][::step] * 1e9)
    except Exception:
        pass

lats, lons, ch4s = map(np.array, (lats, lons, ch4s))
m = np.isfinite(ch4s) & (ch4s > 0)
lats, lons, ch4s = lats[m], lons[m], ch4s[m]

if len(ch4s) == 0:
    raise ValueError('No valid CH₄ pixels found for coverage plotting.')

vmin, vmax = np.nanpercentile(ch4s, [2, 98])

fig, ax = plt.subplots(figsize=(11, 8.5))
_india_gdf.plot(ax=ax, facecolor='#f5f0eb', edgecolor='#555', linewidth=0.8)
sc = ax.scatter(lons, lats, c=ch4s, s=0.3, cmap='plasma', vmin=vmin, vmax=vmax, alpha=0.6)
plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.03).set_label('CH₄ (ppb)')

ax.set_xlim(65, 100)
ax.set_ylim(5, 40)
ax.set_title(
    f'S5P CH₄ coverage over India | {len(files)} granules | median {np.nanmedian(ch4s):.1f} ppb',
    fontsize=12
)
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_OUT_DIR, 'S5P_CH4_India_coverage.png'), dpi=150, bbox_inches='tight')
plt.show()

print('✅ coverage map saved to Drive')